In [ ]:
def main(datasources, start_date, end_date):
    if not isinstance(datasources, dict):
        raise TypeError('datasources must be a dictionary')
    if 'bar1m' not in datasources:
        raise KeyError('datasources must contain bar1m')
    if not isinstance(datasources['bar1m'], str) or not datasources['bar1m'].strip():
        raise ValueError('datasources bar1m must be a non-empty table identifier')
    from f002_moe_dual_cc_train_reference import FEATURES, build_pool_sql, build_feature_sql, month_chunks, frame_to_tensors, fetch_features, _PatchExpert, _CausalBlock, _TCNExpert, _GRUExpert, FactorModel
    import gc
    import os
    import json
    import math
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import dai

    def build_output_pool_sql(table, d0, d1):
        return 'SELECT pd AS date, instrument FROM (' + build_pool_sql(table, d0, d1) + ') p'

    def predict_factor(models, keys, xs, device, batch=1000, smooth=1):
        acc = np.zeros(len(keys), dtype=np.float64)
        for model in models:
            model.eval()
            out = np.empty(len(keys), dtype=np.float32)
            with torch.no_grad():
                for cid, g in keys.groupby('chunk'):
                    rows = g['row'].to_numpy()
                    pos = g.index.to_numpy()
                    xc = xs[int(cid)]
                    for i0 in range(0, len(rows), batch):
                        xb = torch.from_numpy(xc[rows[i0:i0 + batch]].astype(np.float32)).to(device)
                        out[pos[i0:i0 + batch]] = model(xb).cpu().numpy()
            z = pd.Series(out).groupby(keys['d'].to_numpy()).transform(lambda v: (v - v.mean()) / (v.std(ddof=0) + 1e-09))
            acc += z.to_numpy(np.float64)
        res = keys[['instrument', 'd']].copy()
        res['factor'] = acc / len(models)
        if smooth > 1:
            res = res.sort_values(['instrument', 'd']).reset_index(drop=True)
            res['factor'] = res.groupby('instrument')['factor'].transform(lambda s: s.rolling(smooth, min_periods=1).mean())
            res['factor'] = res.groupby('d')['factor'].transform(lambda v: (v - v.mean()) / (v.std(ddof=0) + 1e-09))
        return res

    class _InferCfg:

        def __init__(self, ac):
            self.T = ac['T']
            self.D_MODEL = ac['D_MODEL']
            self.N_HEADS = ac['N_HEADS']
            self.N_LAYERS = ac['N_LAYERS']
            self.PATCH = ac['PATCH']
            self.TCN_KERNEL = ac['TCN_KERNEL']
            self.TCN_DILATIONS = ac['TCN_DILATIONS']
            self.TCN_LEVELS = ac['TCN_LEVELS']
            self.DROPOUT = ac['DROPOUT']
            self.MIN_BARS = ac['MIN_BARS']
            self.MIN_XSEC = ac.get('MIN_XSEC', 0)
            self.SMOOTH_DAYS = ac.get('SMOOTH_DAYS', 1)

    def _find_weights(name):
        SAME_DIRECTORY_WEIGHT_ONLY = True
        if os.path.isabs(name):
            path = os.path.abspath(name)
        else:
            if name != 'f002_weights.json' or os.path.basename(name) != name:
                raise ValueError('relative weight name must be exactly f002_weights.json')
            path = os.path.join(os.getcwd(), 'f002_weights.json')
        if not os.path.isfile(path):
            raise FileNotFoundError('weight file is missing; package weights must be beside the submission notebook: ' + path)
        return path

    def run_inference(dai_mod, datasources, start_date, end_date, weights_name='f002_weights.json'):
        with open(_find_weights(weights_name), encoding='utf-8') as handle:
            payload = json.load(handle)
        if payload.get('features') != FEATURES:
            raise ValueError('weight features do not match f002_moe_dual_cc code')
        cfg = _InferCfg(payload['config'])
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = FactorModel(payload['arch'], cfg).to(device)
        state = {key: torch.tensor(np.array(value, dtype=np.float32)) for key, value in payload['state_dict'].items()}
        model.load_state_dict(state)
        model.eval()
        sd, ed = (str(start_date)[:10], str(end_date)[:10])
        keys, arrays = fetch_features(dai_mod, datasources['bar1m'], sd, ed, cfg, 1)
        factor = predict_factor([model], keys, arrays, device, smooth=cfg.SMOOTH_DAYS)
        out = factor.rename(columns={'d': 'date'})
        out['date'] = pd.to_datetime(out['date'])
        out = out[out['date'].between(start_date, end_date)]
        pool = dai_mod.query(build_output_pool_sql(datasources['bar1m'], sd, ed), filters={'date': [start_date, end_date]}, compression=True).df()
        pool['date'] = pd.to_datetime(pool['date'])
        out = pd.merge(out[['date', 'instrument', 'factor']], pool, on=['date', 'instrument'])
        out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=['factor'])
        return out[['date', 'instrument', 'factor']]
    return run_inference(dai, datasources, start_date, end_date)
